# Unified Pre-DRR Preprocessing Pipeline

Standardizes raw knee CT volumes into a uniform format suitable for **DRR generation** via DiffDRR.

## Pipeline
Load → Resample (0.5mm isotropic) → Orient (LPS) → Bone Window ([-450, 1050] HU) → **Body-Envelope Mask (drop CT table + extra body parts)** → ROI Bone Crop → **Center into Fixed Physical FOV (200mm cube)** → Resize to TARGET_SIZE³ → Save NIfTI

## Datasets (71 volumes)
| Dataset | Count | Format | Source |
|---------|-------|--------|--------|
| Regular VSD (healthy) | 22 | NIfTI | `data/raw/healthy/VSD.{001-023}/` |
| Z-prefix VSD (healthy) | 36 | NIfTI | `data/raw/healthy/VSD.z{001-066}/` |
| Fractured (Ruikar) | 13 | DICOM | `data/raw/fractured/Part{Left,Right}/` |

Output is split per dataset under `data/interim/predrr_lps_256_v1/{healthy,fractured}/`.

## Laterality, Exclusions & Cleaning (decisions + justification)
- **Non-destructive side mapping** — canonical output IDs are read from configs/vsd_laterality_overrides_v1.json; raw files are never renamed. The active map is empty because the focused four-bone audit found no mapping override is required.
- **TKR metal (healthy)** — one affected knee from each of `z050` and `z063` is excluded; current LPS-centroid evidence infers excluded Left and surviving Right, with user approval pending: total knee replacements saturate the
  HU ceiling, masquerade as bone, and occlude the true joint (unrecoverable). Defensive in code — only
  the survivor files are named `Right` and are retained only after the pending user laterality gate.
- **Metal hardware (fractured)** — `Case8` excluded: lodged metallic hardware saturates HU and hides
  the underlying bone shape (unrecoverable) — same policy as the TKR cases. Dropped from train + eval.
- **Scouts (fractured)** — `Case4`, `Case10` excluded: single-slice scout series (< `MIN_Z_SLICES`).
- **Fused cast (fractured)** — `Case3`, `Case16`: a dense cast/holder is topologically fused to the
  limb, so no threshold / connected-component can separate it. Removed manually in 3D Slicer and
  de-specked via `fractured_cast_cleanup.ipynb` → `data/interim/fractured_cast_cleaned/`; this notebook
  re-points those cases at the cleaned NIfTI via the `CAST_CLEANED` override.
- **Body-envelope mask (ALL cases — VSD + fractured)** — step 4b keeps the largest soft-tissue
  connected component (+ fill holes) and zeros everything outside the limb, removing the CT
  table/support slab and any disconnected extra body parts. Air-gap-separated table cases (e.g.
  `Case5`, `Case11`, `Case12`) are cleaned automatically here — no manual work needed.
- **Contralateral-leg contamination (healthy)** — `VSD.z036`: bilateral FOV captured
  contralateral-leg bone (~14–30 k mm³) that passes `GT_THRESH`. Soft-tissue
  `body_envelope_mask` cannot separate both legs (one soft envelope); a bone distance gate
  is unsafe (flags the fibula). Erosion-based leg separation is applied in
  `z036_contralateral_cleanup.ipynb` → `data/interim/healthy_cleaned/`; this notebook
  re-points those cases at the cleaned NIfTI via the `CONTRALATERAL_CLEANED` override.

## Bugs Fixed (from previous versions)
1. **Non-uniform resize** → replaced with uniform resize (no anatomy distortion)
2. **Wrong orientation** → LPS (was LAS)
3. **Per-volume varying scale (pad-to-cube)** → replaced with a **fixed 200mm physical FOV** so
   output spacing/scale and bone occupancy are CONSTANT across healthy and fractured. The old
   per-volume cube inflated long fractured scans, shrinking their axial bone occupancy and
   effective resolution — which would have biased DRR generation and the U-Net vs V-Net comparison.
4. **Flat output folder** → routed into `healthy/` vs `fractured/` subfolders.
5. **Upside-down volumes (S-I)** → detected live via a blob heuristic and flipped to
   femur-on-top at step 7b; all QA panels render superior-on-top through a shared
   `midslices_upright()` helper.

In [ ]:
import os
import gc
import json
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import ndimage

# ============================================================
# Configuration
# ============================================================
PROJECT_ROOT = Path("/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2 _24020059")

# Target parameters
TARGET_SIZE = 256               # voxels per axis, unified across local and HPC
RESAMPLE_SPACING = 0.5          # mm, isotropic resampling target
ORIENTATION = "LPS"             # Left-Posterior-Superior (foundation contract v1)

# Fixed physical field-of-view (FOV)
# Every volume is placed (centered) into a cube of this physical size BEFORE the final
# resize. This guarantees a CONSTANT output spacing and mm-per-voxel across ALL volumes
# (healthy + fractured), so bone occupancy and effective resolution are consistent.
#   - 200mm matches the healthy ±100mm knee crop (vsd_knee_cropping.ipynb) and is the
#     tightest cube that preserves the full healthy knee (≤~199mm Z, ≤~170mm in-plane).
#   - Fractured shaft beyond 200mm is intentionally trimmed so fractured matches the
#     healthy knee window (equalizes occupancy; we only want to capture the knee bone).
FOV_MM = 200.0                  # physical cube side (mm)
FOV_VOXELS = int(round(FOV_MM / RESAMPLE_SPACING))   # cube side in voxels @ resample spacing (=400)
OUTPUT_SPACING = FOV_MM / TARGET_SIZE                 # CONSTANT isotropic output spacing (mm/voxel)

# Bone windowing
HU_MIN = -450
HU_MAX = 1050

# ROI crop parameters
ROI_INTENSITY_THRESHOLD = 0.1   # fraction of normalized range
ROI_CLOSING_RADIUS = 3          # morphological closing ball radius (voxels)
ROI_PAD_MARGIN = 5              # padding voxels around bounding box

# Scout filter (fractured DICOM)
MIN_Z_SLICES = 10

# Exclusions
TKR_LATERALITY_VERIFIED = True  # User-approved; intrinsic fibula/tibia anatomy confirms Right survivors.
TKR_EXCLUSIONS = {("z050", "Left"), ("z063", "Left")}
LATERALITY_OVERRIDE_PATH = PROJECT_ROOT / "configs" / "vsd_laterality_overrides_v1.json"
_laterality_config = json.loads(LATERALITY_OVERRIDE_PATH.read_text(encoding="utf-8"))
LATERALITY_OVERRIDES = _laterality_config["active_overrides"]
LATERALITY_AUDIT_APPROVED = True  # Focused CT-STL-target and 58-knee multibone audits pass.
FRACTURED_SCOUT_EXCLUSIONS = {"Case4", "Case10"}
# Metal-implant exclusions (fractured): metallic hardware lodged in the bone saturates the HU
# ceiling, masquerades as bone in the GT, and occludes the true bone shape underneath (cannot be
# recovered) - same policy as the TKR metal exclusions (z050/z063). Excluded from train + eval.
FRACTURED_METAL_EXCLUSIONS = {"Case8"}

# Paths
HEALTHY_DIR = PROJECT_ROOT / "data" / "raw" / "healthy"
FRACTURED_LEFT = PROJECT_ROOT / "data" / "raw" / "fractured" / "PartLeft"
FRACTURED_RIGHT = PROJECT_ROOT / "data" / "raw" / "fractured" / "PartRight"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "predrr_lps_256_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Target size     : {TARGET_SIZE}³ voxels")
print(f"  Resample spacing: {RESAMPLE_SPACING} mm isotropic")
print(f"  Orientation     : {ORIENTATION}")
print(f"  Bone window     : [{HU_MIN}, {HU_MAX}] HU")
print(f"  Fixed FOV       : {FOV_MM} mm cube ({FOV_VOXELS}³ vox @ {RESAMPLE_SPACING}mm)")
print(f"  Output spacing  : {OUTPUT_SPACING:.4f} mm/voxel (CONSTANT for all volumes)")
print(f"  Output dir      : {OUTPUT_DIR}")

## Orientation Correction — S–I Upright

Many raw knee volumes are **upside-down** along the superior–inferior axis (femur at the bottom).
This step makes the pipeline save every volume **upright** (femur on top).

At the save point, `arr_final = sitk.GetArrayFromImage(...)` is `(z, y, x)` order, so
**axis 0 = S–I** (high index = superior), axis 1 = A–P, axis 2 = L–R. The fix is therefore
`np.flip(arr_final, axis=0)`.

**Detection is live and idempotent.** Orientation is decided *inside* the pipeline at step 7b by
the connected-component blob heuristic `heuristic_flipped(arr)` (femur end = 1 blob, tibia+fibula
end = 2; a volume is FLIPPED when the superior end shows >=2 blobs and the inferior end <=1). The
heuristic runs on the freshly **raw-derived** array (load→resample→orient→window→crop→fov→resize),
so it flags the same cases on every run regardless of what is currently saved in `predrr/`. The 4
AMBIGUOUS cases (and any case the heuristic gets wrong) are pinned in `SI_FLIP_OVERRIDE`.

> Idempotent by construction: `process_single_volume` always reloads from the **raw** sources and
> re-derives orientation from that array — never from the saved (possibly already-corrected) files.
> Re-running the batch regenerates correct upright output regardless of the current `predrr/` state.
> Raw data is never modified. `orientation_validation.ipynb` and its report CSV are inspection-only
> and do **not** drive this step.

In [ ]:
# ============================================================
# Orientation Correction — S-I upright (femur on top)
# ============================================================
# Detection is LIVE and idempotent: heuristic_flipped(arr) runs on the in-pipeline, raw-derived
# array at step 7b (not from any saved file or CSV), so it flags the same cases every run.
# Heuristic (ported from orientation_validation.ipynb, adapted to arr_final (z, y, x) order:
# axis 0 = S-I, high index = superior): counts bone connected-components per axial slice at each
# end; femur end = 1 blob, tibia+fibula end = 2.
#
# Per-volume decision: SI_FLIP_OVERRIDE[vid] if present, else heuristic_flipped(arr).
# WARNING: do NOT drive this from the validation report CSV - that file is rewritten from the
# CURRENT saved-file orientation, which creates a self-undoing loop (once output is upright the
# CSV reports 0 flipped and correction silently switches off).

BONE_THR = 0.45    # normalized-intensity threshold for "bone"
MIN_AREA = 30      # min voxels for a 2D component to count as a blob
END_FRAC = 0.25    # use the top/bottom quarter of the S-I axis as the two ends

# 4 AMBIGUOUS cases (heuristic could not decide) - set True to force an S-I flip,
# False to leave upright. Also the escape hatch for any confident case the heuristic gets wrong.
# Defaults lean upright; confirm visually in the preview cell below before the batch run.
SI_FLIP_OVERRIDE = {
    "VSD_016_Left": False,
    "VSD_016_Right": False,
    "Case14_PartRight": False,
    "Case3_PartLeft": False,
}


def _blobs_in_axial(sl2d):
    """Number of bone components (area > MIN_AREA) in one (y, x) axial slice."""
    mask = ndimage.binary_opening(sl2d > BONE_THR, iterations=1)
    lab, n = ndimage.label(mask)
    if n == 0:
        return 0
    sizes = ndimage.sum(np.ones_like(lab), lab, range(1, n + 1))
    return int((sizes > MIN_AREA).sum())


def end_blob_counts(arr):
    """Median axial blob count at the superior (high axis-0) and inferior (low axis-0) ends."""
    k = arr.shape[0]
    e = max(1, int(round(k * END_FRAC)))
    sup = float(np.median([_blobs_in_axial(arr[kk]) for kk in range(k - e, k)]))
    inf = float(np.median([_blobs_in_axial(arr[kk]) for kk in range(0, e)]))
    return sup, inf


def heuristic_flipped(arr):
    """True only when the volume is confidently upside-down (superior end >=2, inferior <=1)."""
    sup, inf = end_blob_counts(arr)
    return sup >= 2 and inf <= 1


def correct_orientation(vid, arr):
    """Flip the volume S-I upright if needed. Returns (arr, flags).

    Decision: SI_FLIP_OVERRIDE[vid] if pinned, else the live heuristic on this raw-derived array.
    """
    flip_si = SI_FLIP_OVERRIDE.get(vid, heuristic_flipped(arr))
    if flip_si:
        arr = np.ascontiguousarray(np.flip(arr, axis=0))
    return arr, {"si_flipped": bool(flip_si)}


print("Orientation correction defined: correct_orientation(vid, arr)  [S-I upright, live + idempotent]")
print(f"  Heuristic: BONE_THR={BONE_THR}, MIN_AREA={MIN_AREA}, END_FRAC={END_FRAC}")
print(f"  SI_FLIP_OVERRIDE (manual pins): {SI_FLIP_OVERRIDE}")

In [ ]:
def discover_healthy_nifti(healthy_dir, tkr_exclusions, laterality_overrides):
    """Scan healthy NIfTI knee crops. Header-only reads (no pixel data loaded)."""
    cases = []
    for vsd_dir in sorted(healthy_dir.glob("VSD.*")):
        if not vsd_dir.is_dir():
            continue
        for nifti_path in sorted(vsd_dir.glob("VSD_*.nii.gz")):
            # Parse filename: VSD_{sid}_{side}.nii.gz
            stem = nifti_path.stem.replace(".nii", "")  # remove .nii from .nii.gz
            parts = stem.split("_")
            if len(parts) < 3:
                continue
            sid = parts[1]       # e.g., "001", "z001"
            side = parts[2]      # source filename side
            source_volume_id = f"VSD_{sid}_{side}"
            canonical_volume_id = laterality_overrides.get(source_volume_id, source_volume_id)
            canonical_parts = canonical_volume_id.split("_")
            canonical_sid = canonical_parts[1]
            canonical_side = canonical_parts[2]
            if canonical_volume_id != source_volume_id:
                print(f"  [MAP] {source_volume_id} -> {canonical_volume_id} (raw file unchanged)")

            # Apply TKR exclusions
            if (sid, side) in tkr_exclusions:
                print(f"  [EXCL] VSD_{sid}_{side}: TKR exclusion")
                continue

            # Read header only
            reader = sitk.ImageFileReader()
            reader.SetFileName(str(nifti_path))
            reader.ReadImageInformation()
            size_xyz = reader.GetSize()
            spacing_xyz = reader.GetSpacing()

            dataset = "healthy_z" if canonical_sid.startswith("z") else "healthy"
            cases.append({
                "volume_id": canonical_volume_id,
                "source_volume_id": source_volume_id,
                "dataset": dataset,
                "source_format": "nifti",
                "file_path": str(nifti_path),
                "size_xyz": size_xyz,
                "spacing_xyz": tuple(round(s, 6) for s in spacing_xyz),
            })
    return cases


def discover_fractured_dicom(left_dir, right_dir, scout_exclusions, min_z, metal_exclusions=frozenset()):
    """Scan fractured DICOM case directories. Header-only reads."""
    cases = []
    for part_dir, part_name in [(left_dir, "PartLeft"), (right_dir, "PartRight")]:
        if not part_dir.exists():
            print(f"  [WARN] Directory not found: {part_dir}")
            continue
        for case_dir in sorted(part_dir.iterdir()):
            if not case_dir.is_dir():
                continue
            case_id = case_dir.name

            # Apply scout exclusions
            if case_id in scout_exclusions:
                print(f"  [EXCL] {case_id}_{part_name}: scout exclusion")
                continue

            # Apply metal-implant exclusions
            if case_id in metal_exclusions:
                print(f"  [EXCL] {case_id}_{part_name}: metal exclusion")
                continue

            reader = sitk.ImageSeriesReader()
            dicom_names = reader.GetGDCMSeriesFileNames(str(case_dir))
            if not dicom_names:
                continue

            # Read first DICOM header for spacing/size
            file_reader = sitk.ImageFileReader()
            file_reader.SetFileName(dicom_names[0])
            file_reader.ReadImageInformation()
            size_2d = file_reader.GetSize()
            spacing_2d = file_reader.GetSpacing()
            n_slices = len(dicom_names)

            if n_slices < min_z:
                print(f"  [EXCL] {case_id}_{part_name}: only {n_slices} slices (scout)")
                continue

            # Estimate z-spacing
            z_spacing = 0.7  # fallback
            if n_slices >= 2:
                try:
                    pos0 = [float(v) for v in file_reader.GetMetaData("0020|0032").strip().split("\\")]
                    file_reader.SetFileName(dicom_names[1])
                    file_reader.ReadImageInformation()
                    pos1 = [float(v) for v in file_reader.GetMetaData("0020|0032").strip().split("\\")]
                    z_spacing = abs(pos1[2] - pos0[2])
                except Exception:
                    try:
                        z_spacing = float(file_reader.GetMetaData("0018|0050").strip())
                    except Exception:
                        pass

            cases.append({
                "volume_id": f"{case_id}_{part_name}",
                "dataset": "fractured",
                "source_format": "dicom",
                "file_path": str(case_dir),
                "size_xyz": (size_2d[0], size_2d[1], n_slices),
                "spacing_xyz": (round(spacing_2d[0], 6), round(spacing_2d[1], 6), round(z_spacing, 6)),
            })
    return cases


# ============================================================
# Run discovery
# ============================================================
print("Scanning datasets...\n")

print("--- Healthy NIfTI ---")
assert TKR_LATERALITY_VERIFIED, (
    'BLOCKED: approve z050/z063 anatomical sides from LPS-centroid evidence, then populate TKR_EXCLUSIONS.'
)
assert LATERALITY_AUDIT_APPROVED, (
    "BLOCKED: laterality audit approval is required before preprocessing."
)
healthy_cases = discover_healthy_nifti(HEALTHY_DIR, TKR_EXCLUSIONS, LATERALITY_OVERRIDES)
print(f"  Found {len(healthy_cases)} healthy volumes\n")

print("--- Fractured DICOM ---")
fractured_cases = discover_fractured_dicom(
    FRACTURED_LEFT, FRACTURED_RIGHT, FRACTURED_SCOUT_EXCLUSIONS, MIN_Z_SLICES,
    FRACTURED_METAL_EXCLUSIONS
)
print(f"  Found {len(fractured_cases)} fractured volumes\n")

all_cases = healthy_cases + fractured_cases
df_cases = pd.DataFrame(all_cases)

# Summary
print(f"{'='*60}")
print(f"Total volumes: {len(df_cases)}")
for ds in ["healthy", "healthy_z", "fractured"]:
    n = len(df_cases[df_cases["dataset"] == ds])
    print(f"  {ds:12s}: {n}")

display(df_cases[["volume_id", "dataset", "source_format", "size_xyz", "spacing_xyz"]])

In [ ]:
# ============================================================
# Cast-cleaned source override (fused-cast cases only)
# ============================================================
# Case3 and Case16 have a cast topologically FUSED to the limb (cannot be auto-separated by
# body_envelope_mask). They were removed manually in 3D Slicer + de-specked via
# fractured_cast_cleanup.ipynb -> data/interim/fractured_cast_cleaned/<Case>_clean.nii.gz.
# Re-point those cases at the cleaned NIfTI so their predrr no longer rebuilds the cast from
# the raw DICOM. (Case5/Case11 are TABLE, not cast -- air-gap separated -- so they stay on
# raw DICOM and are cleaned automatically by body_envelope_mask during processing.)
CAST_CLEANED = {
    "Case3_PartLeft": "Case3_clean.nii.gz",
    "Case16_PartRight": "Case16_clean.nii.gz",
}
_clean_dir = PROJECT_ROOT / "data" / "interim" / "fractured_cast_cleaned"
for vid, fname in CAST_CLEANED.items():
    p = _clean_dir / fname
    m = df_cases["volume_id"] == vid
    if m.any() and p.exists():
        df_cases.loc[m, "source_format"] = "nifti"
        df_cases.loc[m, "file_path"] = str(p)
        print(f"  [CAST-CLEANED] {vid} -> {p.name}")
    elif m.any():
        print(f"  [WARN] {vid}: cleaned file missing ({p}); still using raw DICOM (cast present)")
    else:
        print(f"  [WARN] {vid}: not found in df_cases (skipped)")

# ============================================================
# Contralateral-leg cleaned source override (z036 only)
# ============================================================
# VSD.z036 is a bilateral full-body CT whose knee crop captured contralateral-leg bone inside the
# 200mm FOV. The soft-tissue body_envelope_mask cannot separate two legs sharing one envelope, and
# a bone-level distance gate is unsafe (flags the fibula). Erosion-based leg separation is applied
# in z036_contralateral_cleanup.ipynb -> data/interim/healthy_cleaned/.
# z036 Right: auto-separated (erosion breaks the inter-leg bridge cleanly).
# z036 Left : bridged even after erosion -> manual 3D Slicer step required; warn until file exists.
CONTRALATERAL_CLEANED = {
    "VSD_z036_Right": "VSD_z036_Right_clean.nii.gz",
    "VSD_z036_Left":  "VSD_z036_Left_clean.nii.gz",
}
_healthy_clean_dir = PROJECT_ROOT / "data" / "interim" / "healthy_cleaned"
for vid, fname in CONTRALATERAL_CLEANED.items():
    p = _healthy_clean_dir / fname
    m = df_cases["volume_id"] == vid
    if m.any() and p.exists():
        df_cases.loc[m, "source_format"] = "nifti"
        df_cases.loc[m, "file_path"] = str(p)
        print(f"  [CONTRALATERAL-CLEANED] {vid} -> {p.name}")
    elif m.any():
        print(f"  [WARN] {vid}: cleaned file missing ({p}); still using raw NIfTI (contralateral bone present)")
    else:
        print(f"  [WARN] {vid}: not found in df_cases (skipped)")


In [ ]:
# ============================================================
# Pipeline Helper Functions
# ============================================================

def load_volume(row):
    """Unified volume loader — handles both NIfTI and DICOM."""
    if row["source_format"] == "nifti":
        return sitk.ReadImage(row["file_path"])
    else:
        reader = sitk.ImageSeriesReader()
        dicom_names = reader.GetGDCMSeriesFileNames(row["file_path"])
        reader.SetFileNames(dicom_names)
        return reader.Execute()


def resample_volume(sitk_img, new_spacing=(RESAMPLE_SPACING,)*3):
    """Resample a SimpleITK image to isotropic spacing."""
    original_spacing = sitk_img.GetSpacing()
    original_size = sitk_img.GetSize()

    new_size = [
        int(round(osz * osp / nsp))
        for osz, osp, nsp in zip(original_size, original_spacing, new_spacing)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputDirection(sitk_img.GetDirection())
    resampler.SetOutputOrigin(sitk_img.GetOrigin())
    resampler.SetTransform(sitk.Transform())
    resampler.SetDefaultPixelValue(float(sitk.GetArrayViewFromImage(sitk_img).min()))
    resampler.SetInterpolator(sitk.sitkLinear)
    return resampler.Execute(sitk_img)


def orient_volume(sitk_img, orientation=ORIENTATION):
    """Reorient a SimpleITK image to the target orientation."""
    orienter = sitk.DICOMOrientImageFilter()
    orienter.SetDesiredCoordinateOrientation(orientation)
    return orienter.Execute(sitk_img)


def apply_bone_window(arr, hu_min=HU_MIN, hu_max=HU_MAX):
    """Clip to bone window and normalize to [0, 1]."""
    arr = np.clip(arr, hu_min, hu_max)
    arr = (arr - hu_min) / (hu_max - hu_min)
    return arr.astype(np.float32)


def roi_bone_crop(arr_windowed, threshold=ROI_INTENSITY_THRESHOLD,
                  closing_radius=ROI_CLOSING_RADIUS, pad=ROI_PAD_MARGIN):
    """
    Tight crop around bone anatomy.
    1. Threshold windowed volume to create binary mask
    2. Morphological closing to fill gaps
    3. Keep largest connected component
    4. Bounding box + padding
    """
    mask = (arr_windowed > threshold).astype(np.uint8)

    struct = ndimage.generate_binary_structure(3, 1)
    struct = ndimage.iterate_structure(struct, closing_radius)
    mask = ndimage.binary_closing(mask, structure=struct).astype(np.uint8)

    labeled, num_features = ndimage.label(mask)
    if num_features == 0:
        print("  [WARN] No foreground detected, returning full volume.")
        return arr_windowed, arr_windowed.shape

    component_sizes = ndimage.sum(mask, labeled, range(1, num_features + 1))
    largest_label = np.argmax(component_sizes) + 1
    mask = (labeled == largest_label).astype(np.uint8)

    coords = np.argwhere(mask)
    z_min, y_min, x_min = coords.min(axis=0)
    z_max, y_max, x_max = coords.max(axis=0) + 1

    z_min = max(0, z_min - pad)
    y_min = max(0, y_min - pad)
    x_min = max(0, x_min - pad)
    z_max = min(arr_windowed.shape[0], z_max + pad)
    y_max = min(arr_windowed.shape[1], y_max + pad)
    x_max = min(arr_windowed.shape[2], x_max + pad)

    cropped = arr_windowed[z_min:z_max, y_min:y_max, x_min:x_max]
    return cropped, cropped.shape


def body_envelope_mask(arr_windowed, soft_thr=ROI_INTENSITY_THRESHOLD, open_iter=1):
    """Zero everything outside the limb. The CT table/support slab is separated from the leg by
    an air gap, so at a soft-tissue threshold it forms a separate connected component; keeping
    the largest component (+ fill holes) drops the slab while preserving bone, soft tissue and
    any displaced fracture fragment inside the skin envelope. Orientation-agnostic."""
    body = arr_windowed > soft_thr
    struct = ndimage.generate_binary_structure(3, 1)
    body = ndimage.binary_opening(body, struct, iterations=open_iter)
    labeled, n = ndimage.label(body)
    if n == 0:
        return arr_windowed
    sizes = ndimage.sum(np.ones_like(labeled), labeled, range(1, n + 1))
    keep = ndimage.binary_fill_holes(labeled == (np.argmax(sizes) + 1))
    return np.where(keep, arr_windowed, 0.0).astype(np.float32)


def center_to_fixed_fov(arr, fov_voxels=FOV_VOXELS, fill_value=0.0):
    """
    Place the (bone-tight) ROI crop, CENTERED, into a fixed cubic box of side `fov_voxels`.

    This replaces the old pad_to_cube (which sized the cube to the longest crop axis and so
    produced a different physical FOV / spacing per volume). With a FIXED box:
      - axes SHORTER than fov_voxels are center-padded with `fill_value` (black)
      - axes LONGER  than fov_voxels are center-cropped (excess anatomy trimmed)
    => every volume shares the same physical FOV and (after resize) the same mm/voxel.

    Centering uses the array center, which equals the ROI-crop center (roi_bone_crop already
    tightened the box around the largest bone component).

    Returns: (fixed_box_array, clipped_flags) where clipped_flags = (clip_z, clip_y, clip_x),
    each True if that axis was longer than fov_voxels (i.e. anatomy was trimmed).
    """
    out = np.full((fov_voxels, fov_voxels, fov_voxels), fill_value, dtype=arr.dtype)
    clipped = [False, False, False]

    src_slices, dst_slices = [], []
    for axis, n in enumerate(arr.shape):
        if n <= fov_voxels:
            # center-pad: full source fits, place it centered in the box
            start = (fov_voxels - n) // 2
            src_slices.append(slice(0, n))
            dst_slices.append(slice(start, start + n))
        else:
            # center-crop: take the central fov_voxels of the source
            start = (n - fov_voxels) // 2
            src_slices.append(slice(start, start + fov_voxels))
            dst_slices.append(slice(0, fov_voxels))
            clipped[axis] = True

    out[tuple(dst_slices)] = arr[tuple(src_slices)]
    return out, tuple(clipped)


def resize_volume(arr, target_size=TARGET_SIZE):
    """Resize volume to target cubic size using trilinear interpolation."""
    sitk_img = sitk.GetImageFromArray(arr)
    target = [target_size, target_size, target_size]

    original_size = sitk_img.GetSize()
    original_spacing = sitk_img.GetSpacing()

    new_spacing = [
        original_spacing[i] * original_size[i] / target[i]
        for i in range(3)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(target)
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetOutputOrigin(sitk_img.GetOrigin())
    resampler.SetOutputDirection(sitk_img.GetDirection())
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetDefaultPixelValue(0.0)
    resampler.SetTransform(sitk.Transform())

    resized_img = resampler.Execute(sitk_img)
    return sitk.GetArrayFromImage(resized_img).astype(np.float32)


# Output spacing is now CONSTANT for every volume (fixed FOV / target size).
def compute_output_spacing(fov_mm=FOV_MM, target_size=TARGET_SIZE):
    """Constant isotropic output spacing after fixed-FOV box + resize."""
    return fov_mm / target_size


def midslices_upright(arr):
    """(axial, coronal, sagittal) midslices with SUPERIOR on top.

    arr is a SimpleITK array in (z, y, x) order: axis 0 = S-I (index 0 = inferior),
    axis 1 = A-P, axis 2 = L-R. matplotlib's imshow defaults to origin='upper', so the
    coronal/sagittal panels must be flipped along axis 0 to render superior-on-top.
    Axial (arr[mz]) is an A-P x L-R slice and needs no S-I flip.
    """
    mz, my, mx = (s // 2 for s in arr.shape)
    return arr[mz], arr[::-1, my, :], arr[::-1, :, mx]


print("Helper functions defined:")
print("  load_volume, resample_volume, orient_volume, apply_bone_window,")
print("  body_envelope_mask, roi_bone_crop, center_to_fixed_fov, resize_volume, compute_output_spacing, midslices_upright")
print(f"  Fixed FOV box: {FOV_VOXELS}³ vox → resize {TARGET_SIZE}³ → spacing {compute_output_spacing():.4f}mm (constant)")

In [ ]:
# ============================================================
# Demo: Single Case Walkthrough
# ============================================================
# Walk one healthy + one fractured case through every pipeline step

demo_rows = []
for ds in ["healthy", "fractured"]:
    subset = df_cases[df_cases["dataset"] == ds]
    if len(subset) > 0:
        demo_rows.append(subset.iloc[0])

for demo_row in demo_rows:
    vid = demo_row["volume_id"]
    print(f"\n{'='*60}")
    print(f"Demo: {vid} ({demo_row['dataset']}, {demo_row['source_format']})")
    print(f"{'='*60}")

    # 1. Load
    img = load_volume(demo_row)
    print(f"\n1. Load:      size={img.GetSize()}, spacing={tuple(round(s,4) for s in img.GetSpacing())}")

    # 2. Resample to 0.5mm isotropic
    img = resample_volume(img)
    print(f"2. Resample:  size={img.GetSize()}, spacing={img.GetSpacing()}")

    # 3. Orient to LPS
    img = orient_volume(img)
    print(f"3. Orient:    size={img.GetSize()}, orientation={ORIENTATION}")

    # 4. Bone windowing
    arr = sitk.GetArrayFromImage(img).astype(np.float32)
    del img
    arr = apply_bone_window(arr)
    print(f"4. Window:    shape={arr.shape}, range=[{arr.min():.3f}, {arr.max():.3f}]")

    # 4b. Body-envelope mask (remove CT table/support slab before crop)
    _occ_before = float((arr > ROI_INTENSITY_THRESHOLD).mean())
    arr = body_envelope_mask(arr)
    _occ_after = float((arr > ROI_INTENSITY_THRESHOLD).mean())
    print(f"4b. Envelope: shape={arr.shape}, soft-tissue occ {_occ_before:.1%} → {_occ_after:.1%}")

    # 5. ROI crop
    arr, crop_shape = roi_bone_crop(arr)
    print(f"5. ROI crop:  shape={arr.shape}")

    # 6. Center into fixed FOV box
    arr, clipped = center_to_fixed_fov(arr, FOV_VOXELS)
    print(f"6. Fixed FOV: shape={arr.shape}, clipped(z,y,x)={clipped}")

    # 7. Resize to TARGET_SIZE
    arr = resize_volume(arr)
    print(f"7. Resize:    shape={arr.shape}")

    # 8. Output spacing (constant)
    output_spacing = compute_output_spacing()
    phys_fov = output_spacing * TARGET_SIZE
    print(f"8. Spacing:   {output_spacing:.4f} mm isotropic, FOV = {phys_fov:.1f} mm (constant)")

    # Visualize final volume — 3-plane views
    mid = [s // 2 for s in arr.shape]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(arr[mid[0], :, :], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title(f"Axial (z={mid[0]})")
    axes[1].imshow(arr[:, mid[1], :], cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(f"Coronal (y={mid[1]})")
    axes[2].imshow(arr[:, :, mid[2]], cmap="gray", vmin=0, vmax=1)
    axes[2].set_title(f"Sagittal (x={mid[2]})")
    for ax in axes:
        ax.set_aspect("equal")
    plt.suptitle(f"{vid} — Final {TARGET_SIZE}³ @ {output_spacing:.3f}mm (fixed {FOV_MM:.0f}mm FOV)", fontsize=13)
    plt.tight_layout()
    plt.show()

    del arr
    gc.collect()

In [ ]:
def process_single_volume(row):
    """
    Full preprocessing pipeline for one volume:
    Load -> Resample -> Orient -> Window -> Body-Envelope Mask -> ROI Crop -> Center to Fixed FOV -> Resize

    Returns: (arr_final, metadata_dict, output_spacing)
    """
    vid = row["volume_id"]

    # 1. Load
    img = load_volume(row)
    orig_size = img.GetSize()
    orig_spacing = img.GetSpacing()
    raw_arr = sitk.GetArrayFromImage(img)
    hu_min_raw, hu_max_raw = float(raw_arr.min()), float(raw_arr.max())
    del raw_arr

    # 2. Resample to isotropic
    img = resample_volume(img)
    resampled_size = img.GetSize()

    # 3. Orient
    img = orient_volume(img)

    # 4. Bone windowing
    arr = sitk.GetArrayFromImage(img).astype(np.float32)
    del img; gc.collect()
    arr = apply_bone_window(arr)

    # 4b. Body-envelope mask (remove CT table/support slab before bone crop)
    _bone_before = int((arr > 0.40).sum())
    arr = body_envelope_mask(arr)
    _envelope_retention = int((arr > 0.40).sum()) / max(_bone_before, 1)

    # 5. ROI crop (bone-tight)
    arr, crop_shape = roi_bone_crop(arr)

    # 6. Center into a FIXED physical FOV box (constant size/scale across all volumes)
    arr, clipped = center_to_fixed_fov(arr, FOV_VOXELS)

    # 7. Resize fixed box to target
    arr = resize_volume(arr, TARGET_SIZE)

    # 7b. Orientation correction - make the volume S-I upright (femur on top)
    arr, ori_flags = correct_orientation(vid, arr)

    # 8. Output spacing is constant (FOV / target size)
    output_spacing = compute_output_spacing()

    metadata = {
        "volume_id": vid,
        "dataset": row["dataset"],
        "source_format": row["source_format"],
        "si_flipped": ori_flags["si_flipped"],
        "orig_size": str(orig_size),
        "orig_spacing": str(tuple(round(s, 6) for s in orig_spacing)),
        "resampled_size": str(resampled_size),
        "crop_shape": str(crop_shape),
        "fov_mm": FOV_MM,
        "fov_voxels": FOV_VOXELS,
        "clipped_z": bool(clipped[0]),
        "clipped_y": bool(clipped[1]),
        "clipped_x": bool(clipped[2]),
        "final_shape": str(arr.shape),
        "output_spacing_mm": round(output_spacing, 6),
        "phys_fov_mm": round(output_spacing * TARGET_SIZE, 1),
        "hu_min_raw": hu_min_raw,
        "hu_max_raw": hu_max_raw,
        "intensity_min": float(arr.min()),
        "intensity_max": float(arr.max()),
        "envelope_retention": round(_envelope_retention, 4),
        "source_path": row["file_path"],
    }

    return arr, metadata, output_spacing


print("Pipeline orchestration function defined: process_single_volume(row)")

In [ ]:
# ============================================================
# Orientation Preview - BEFORE vs AFTER correction (run before the batch loop)
# ============================================================
# Processes a sample of flagged + ambiguous + control cases through the FULL pipeline (incl.
# correct_orientation) and shows BEFORE vs AFTER coronal & sagittal, superior on TOP. Confirm
# every "after" shows femur-on-top and finalize SI_FLIP_OVERRIDE for the ambiguous cases.
# NOTE: a pure S-I flip is obvious in BOTH coronal and sagittal (the femur/tibia swap ends).

_controls = ["VSD_002_Right", "Case11_PartRight"]      # known-upright controls
_ambig = list(SI_FLIP_OVERRIDE.keys())                  # 4 ambiguous cases
_rp_path = OUTPUT_DIR / "orientation_validation_report.csv"
_flipped = []
if _rp_path.exists():
    _rp = pd.read_csv(_rp_path)
    _flipped = _rp[_rp["verdict"] == "FLIPPED"]["case"].tolist()[:3]   # a few examples

_preview_ids, _seen = [], set()
for v in _flipped + _ambig + _controls:
    if v not in _seen:
        _preview_ids.append(v); _seen.add(v)

def _cor(a):   # coronal mid-slice, superior on top (shared convention)
    return midslices_upright(a)[1]
def _sag(a):   # sagittal mid-slice, superior on top (shared convention)
    return midslices_upright(a)[2]

_rows = []
for vid in _preview_ids:
    _r = df_cases[df_cases["volume_id"] == vid]
    if _r.empty:
        print(f"[skip] {vid} not in df_cases")
        continue
    _after, _meta, _ = process_single_volume(_r.iloc[0])
    _flip = _meta["si_flipped"]
    _before = np.flip(_after, axis=0) if _flip else _after
    _rows.append((vid, _flip, _before, _after))

fig, axes = plt.subplots(len(_rows), 4, figsize=(12, 3 * len(_rows)))
if len(_rows) == 1:
    axes = axes[np.newaxis, :]
for i, (vid, flip, b, a) in enumerate(_rows):
    for j, (t, img) in enumerate([("before coronal", _cor(b)), ("before sagittal", _sag(b)),
                                   ("after coronal", _cor(a)), ("after sagittal", _sag(a))]):
        axes[i, j].imshow(img, cmap="gray", vmin=0, vmax=1, aspect="equal")
        axes[i, j].set_title(t, fontsize=8)
        axes[i, j].axis("off")
    _amb = vid in SI_FLIP_OVERRIDE
    _tag = "FLIP" if flip else "no-flip"
    _col = "red" if flip else ("darkorange" if _amb else "green")
    axes[i, 0].text(-0.18, 0.5, f"{vid}\n{_tag}{' [AMBIG]' if _amb else ''}",
                    transform=axes[i, 0].transAxes, fontsize=8, va="center", ha="right", color=_col)
fig.suptitle("Orientation preview - 'after' should show femur on TOP. "
             "Ambiguous (orange): set SI_FLIP_OVERRIDE if 'after' is upside-down.", fontsize=11)
plt.tight_layout()
plt.show()
gc.collect()

In [ ]:
# ============================================================
# Batch Processing — All Volumes
# ============================================================
all_metadata = []
processed_count = 0
error_cases = []

print(f"Processing {len(df_cases)} volumes...\n")

for idx, row in df_cases.iterrows():
    vid = row["volume_id"]
    print(f"[{idx+1}/{len(df_cases)}] {vid}...", end=" ")

    try:
        arr_final, meta, output_spacing = process_single_volume(row)

        # Route into healthy/ or fractured/ subfolder
        subfolder = "healthy" if row["dataset"] in ("healthy", "healthy_z") else "fractured"
        case_out_dir = OUTPUT_DIR / subfolder
        case_out_dir.mkdir(parents=True, exist_ok=True)
        out_name = f"{vid}.nii.gz"
        out_path = case_out_dir / out_name

        nifti_img = sitk.GetImageFromArray(arr_final)
        nifti_img.SetSpacing((output_spacing, output_spacing, output_spacing))
        sitk.WriteImage(nifti_img, str(out_path))

        meta["output_path"] = str(out_path)
        all_metadata.append(meta)
        processed_count += 1
        print(f"✅ → {subfolder}/{out_name} (spacing={output_spacing:.4f}mm)")

        del arr_final, nifti_img
        gc.collect()

    except Exception as e:
        print(f"❌ Error: {e}")
        error_cases.append({"volume_id": vid, "error": str(e)})

# Save metadata CSV at top level
df_meta = pd.DataFrame(all_metadata)
meta_path = OUTPUT_DIR / "preprocessing_metadata.csv"
df_meta.to_csv(meta_path, index=False)

print(f"\n{'='*60}")
print(f"Done. {processed_count}/{len(df_cases)} volumes processed.")
print(f"Output layout: {OUTPUT_DIR}/healthy/ and {OUTPUT_DIR}/fractured/")
print(f"Metadata: {meta_path}")

if error_cases:
    print(f"\n⚠️  {len(error_cases)} case(s) failed:")
    display(pd.DataFrame(error_cases))

In [ ]:
# ============================================================
# QA: Body-Envelope Mask — Retention Check & Residual Column Audit
# ============================================================
# Validates that body_envelope_mask (step 4b) in process_single_volume:
#   (a) did not over-remove bone in any case (retention < 0.95 → flag)
#   (b) removed all residual full-height peripheral artifact columns (slab / table)
#   (c) visual: before/after 3-plane for Case12_PartRight (worst affected fractured case)
GT_THRESH = 0.40   # bone-voxel threshold (matches decoder pipeline)

df_meta_qa = pd.read_csv(OUTPUT_DIR / "preprocessing_metadata.csv")

# ---- (a) Retention check (catches over-removal of bone) ----
print("(a) Envelope-mask bone retention per case (flag if < 0.95):\n")
flagged_ret = []
for _, m in df_meta_qa.iterrows():
    ret = m.get("envelope_retention", None)
    if ret is None or (isinstance(ret, float) and np.isnan(ret)):
        print(f"  ⚠️  {m['volume_id']}: envelope_retention not in metadata (re-run batch?)")
        continue
    tag = "✅" if ret >= 0.95 else "❌ FLAGGED"
    if ret < 0.95:
        flagged_ret.append(m["volume_id"])
    if ret < 0.99:   # only print non-trivial cases to keep output compact
        print(f"  {tag}  {m['volume_id']:35s}  retention={ret:.4f}")

print(f"\nTotal flagged (< 0.95): {len(flagged_ret)}")
if flagged_ret:
    print("  → Review these volumes — the mask may have removed displaced bone:")
    for vid in flagged_ret:
        print(f"       {vid}")
else:
    print("  ✅ No cases flagged — bone retention ≥ 0.95 for all volumes.")

# ---- (b) Residual peripheral full-height column check (catches under-removal = slab still present) ----
print("\n(b) Residual full-height peripheral column check (slab artifact should be gone):\n")
residual_counts = {}
for _, m in df_meta_qa.iterrows():
    vid = m["volume_id"]
    subfolder = "healthy" if m["dataset"] in ("healthy", "healthy_z") else "fractured"
    fpath = OUTPUT_DIR / subfolder / f"{vid}.nii.gz"
    if not fpath.exists():
        continue
    arr = sitk.GetArrayFromImage(sitk.ReadImage(str(fpath)))
    H = arr.shape[0]
    bone = arr > GT_THRESH
    xs = np.where(bone)[2]
    cx = int(xs.mean()) if xs.size > 0 else arr.shape[2] // 2
    col_heights = bone.sum(axis=0).max(axis=0)   # max bone height across y for each x
    periph_mask = np.abs(np.arange(arr.shape[2]) - cx) > 50
    n_artifact = int(((col_heights > 0.6 * H) & periph_mask).sum())
    residual_counts[vid] = n_artifact
    del arr

n_with_cols = sum(1 for v in residual_counts.values() if v > 0)
print(f"Volumes with residual full-height peripheral columns: {n_with_cols}/{len(residual_counts)}")
if n_with_cols:
    for vid, cnt in residual_counts.items():
        if cnt > 0:
            print(f"  ⚠️  {vid}: {cnt} column(s)")
else:
    print("  ✅ None — body-envelope mask removed all lateral slabs.")

# ---- (c) Visual: before/after for flagged + known problem fractured cases ----
_problem = ["Case3_PartLeft", "Case16_PartRight", "Case5_PartRight", "Case11_PartRight", "Case12_PartRight"]
vis_ids = [v for v in dict.fromkeys(_problem + list(flagged_ret))
           if (df_cases["volume_id"] == v).any()]
print(f"\n(c) Before/after body-envelope mask for {len(vis_ids)} case(s): {vis_ids}")
for vid in vis_ids:
    row = df_cases[df_cases["volume_id"] == vid].iloc[0]
    _img = orient_volume(resample_volume(load_volume(row)))
    _raw = apply_bone_window(sitk.GetArrayFromImage(_img).astype(np.float32))
    del _img
    _msk = body_envelope_mask(_raw)
    mid = [s // 2 for s in _raw.shape]
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    praw = [_raw[mid[0]], _raw[:, mid[1], :], _raw[:, :, mid[2]]]
    pmsk = [_msk[mid[0]], _msk[:, mid[1], :], _msk[:, :, mid[2]]]
    for j, (t, pr, pm) in enumerate(zip(["Axial", "Coronal", "Sagittal"], praw, pmsk)):
        axes[0, j].imshow(pr, cmap="gray", vmin=0, vmax=1); axes[0, j].set_title(f"BEFORE - {t}", fontsize=9)
        axes[1, j].imshow(pm, cmap="gray", vmin=0, vmax=1); axes[1, j].set_title(f"AFTER - {t}", fontsize=9)
        axes[0, j].axis("off"); axes[1, j].axis("off")
    _ret = df_meta_qa.loc[df_meta_qa["volume_id"] == vid, "envelope_retention"]
    _rtxt = f"retention={_ret.iloc[0]:.3f}" if len(_ret) else "retention=n/a"
    plt.suptitle(f"{vid}: body-envelope mask BEFORE / AFTER  ({_rtxt})\n"
                 "(table/slab should vanish in AFTER; bone + fragments must remain)", fontsize=11)
    plt.tight_layout(); plt.show()
    del _raw, _msk
    gc.collect()


In [ ]:
# ============================================================
# Validation: Shape, Spacing, Intensity
# ============================================================
print("Validating saved volumes...\n")

saved_files = sorted(OUTPUT_DIR.glob("**/*.nii.gz"))  # recursive — covers healthy/ and fractured/
val_records = []

expected_shape = (TARGET_SIZE, TARGET_SIZE, TARGET_SIZE)
all_ok = True

for fpath in saved_files:
    vol = sitk.ReadImage(str(fpath))
    arr = sitk.GetArrayFromImage(vol)
    sp = vol.GetSpacing()

    subfolder = fpath.parent.name  # "healthy" or "fractured"
    shape_ok = arr.shape == expected_shape
    spacing_iso = abs(sp[0] - sp[1]) < 1e-6 and abs(sp[1] - sp[2]) < 1e-6
    # NEW: spacing must equal the single constant OUTPUT_SPACING across every volume
    spacing_const = all(abs(s - OUTPUT_SPACING) < 1e-4 for s in sp)
    range_ok = arr.min() >= -0.01 and arr.max() <= 1.01
    no_nan = not np.any(np.isnan(arr))

    ok = shape_ok and spacing_iso and spacing_const and range_ok and no_nan
    if not ok:
        all_ok = False

    val_records.append({
        "file": f"{subfolder}/{fpath.name}",
        "shape": arr.shape,
        "spacing": tuple(round(s, 4) for s in sp),
        "isotropic": "✓" if spacing_iso else "✗",
        "const_spacing": "✓" if spacing_const else "✗",
        "range": f"[{arr.min():.3f}, {arr.max():.3f}]",
        "valid": "✅" if ok else "❌",
    })

df_val = pd.DataFrame(val_records)
display(df_val)

healthy_count = sum(1 for r in val_records if r["file"].startswith("healthy/"))
fractured_count = sum(1 for r in val_records if r["file"].startswith("fractured/"))
print(f"\nTotal files : {len(saved_files)}  (healthy: {healthy_count}, fractured: {fractured_count})")
print(f"All shapes {expected_shape}: {'✅ Yes' if all(r['shape'] == expected_shape for r in val_records) else '❌ No'}")
print(f"All isotropic: {'✅ Yes' if all(r['isotropic'] == '✓' for r in val_records) else '❌ No'}")
print(f"All spacing == {OUTPUT_SPACING:.4f}mm (constant): "
      f"{'✅ Yes' if all(r['const_spacing'] == '✓' for r in val_records) else '❌ No'}")
print(f"All valid: {'✅ Yes' if all_ok else '❌ Some failures — check table above'}")

## Visual Inspection - All-Planes Review

One static figure per dataset group (healthy / healthy_z / fractured): one row per volume,
columns = **Axial | Coronal | Sagittal** midslice, rendered **superior-on-top** via the shared
`midslices_upright()` helper. Each row shows a short ID and axial bone-occupancy % for a quick
scan (black frames, mis-crops, clipped joints). Memory-efficient: one 3-D volume in RAM at a time.

In [ ]:
# ============================================================
# Visual Inspection - All Cases, All 3 Planes (superior-on-top)
# ============================================================
# One row per volume: Axial | Coronal | Sagittal midslice, rendered via midslices_upright()
# so the femur sits at the TOP in coronal/sagittal. Three figures (healthy / healthy_z /
# fractured). Memory-efficient: one 3-D volume in RAM at a time. Each row is labelled with a
# short ID and axial bone-occupancy % (quick scan for black frames, mis-crops, clipped joints).
# Replaces the former separate "Spot Check" + "Axial Gallery" + "3-Plane" cells.
df_meta = pd.read_csv(OUTPUT_DIR / "preprocessing_metadata.csv")

PLANES = ["Axial (mid-Z)", "Coronal (sup-top)", "Sagittal (sup-top)"]

for ds_group, ds_label, color in [
    (["healthy"],   "Healthy VSD (regular)",   "steelblue"),
    (["healthy_z"], "Healthy VSD (z-prefix)",  "seagreen"),
    (["fractured"], "Fractured (Ruikar)",       "coral"),
]:
    subset = df_meta[df_meta["dataset"].isin(ds_group)].reset_index(drop=True)
    n = len(subset)
    if n == 0:
        continue

    print(f"Rendering {ds_label} ({n} volumes) ...", end=" ", flush=True)

    fig, axes = plt.subplots(
        n, 3,
        figsize=(9, n * 2.0),
        gridspec_kw={"wspace": 0.02, "hspace": 0.05},
    )
    if n == 1:
        axes = axes[np.newaxis, :]

    # Pre-disable every axis (black face) so a mid-loop read failure leaves no ghost ticks.
    for ax_row in axes:
        for ax in ax_row:
            ax.axis("off")
            ax.set_facecolor("black")

    for j, plane in enumerate(PLANES):
        axes[0, j].set_title(plane, fontsize=9, color="dimgray", pad=3)

    for i, (_, row) in enumerate(subset.iterrows()):
        vid       = row["volume_id"]
        subfolder = "healthy" if row["dataset"] in ("healthy", "healthy_z") else "fractured"
        fpath     = OUTPUT_DIR / subfolder / f"{vid}.nii.gz"

        if not fpath.exists():
            axes[i, 1].text(0.5, 0.5, f"FILE MISSING\n{vid}",
                            transform=axes[i, 1].transAxes,
                            ha="center", va="center", fontsize=7, color="red")
            continue

        # load -> occupancy + 3 superior-on-top midslices -> release 3-D array
        arr    = sitk.GetArrayFromImage(sitk.ReadImage(str(fpath)))
        occ    = float((arr[arr.shape[0] // 2] > ROI_INTENSITY_THRESHOLD).mean())
        panels = midslices_upright(arr)
        del arr

        for j, sl in enumerate(panels):
            axes[i, j].imshow(sl, cmap="gray", vmin=0, vmax=1)

        short_id = (
            vid.replace("VSD_z", "z").replace("VSD_", "")
               .replace("_PartLeft", " L").replace("_PartRight", " R")
               .replace("_Left",     " L").replace("_Right",    " R")
        )
        axes[i, 0].text(
            -0.12, 0.5, f"{short_id}\n{occ:.0%}",
            transform=axes[i, 0].transAxes,
            fontsize=7, va="center", ha="right", clip_on=False,
        )

    fig.suptitle(
        f"{ds_label}  -  Axial | Coronal | Sagittal (superior-on-top)  ({n} volumes)"
        f"  |  {TARGET_SIZE}³ @ {OUTPUT_SPACING:.3f} mm",
        fontsize=11, color=color,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.998])
    plt.show()
    plt.close(fig)
    gc.collect()
    print("done.")

In [ ]:
# ============================================================
# Diagnostic: Bone Occupancy & Scale Consistency (healthy vs fractured)
# ============================================================
# Confirms the fix: after the fixed-FOV pipeline, fractured and healthy volumes should have
# (a) comparable axial bone occupancy, (b) identical output spacing / FOV, and
# (c) no fractured knee joint clipped out by the 200mm trim.

OCC_THR = ROI_INTENSITY_THRESHOLD  # bone = normalized intensity above this

def _load_final(vid, dataset):
    sub = "healthy" if dataset in ("healthy", "healthy_z") else "fractured"
    vol = sitk.ReadImage(str(OUTPUT_DIR / sub / f"{vid}.nii.gz"))
    return sitk.GetArrayFromImage(vol), vol.GetSpacing()

df_meta = pd.read_csv(OUTPUT_DIR / "preprocessing_metadata.csv")

# --- Per-volume occupancy metrics ---
occ_records = []
for _, m in df_meta.iterrows():
    arr, sp = _load_final(m["volume_id"], m["dataset"])
    mid = arr.shape[0] // 2
    occ_records.append({
        "volume_id": m["volume_id"],
        "dataset": m["dataset"],
        "axial_occ": float((arr[mid] > OCC_THR).mean()),
        "vol_occ": float((arr > OCC_THR).mean()),
        "spacing_mm": round(sp[0], 4),
        "phys_fov_mm": m["phys_fov_mm"],
        "clipped": bool(m.get("clipped_z", False) or m.get("clipped_y", False) or m.get("clipped_x", False)),
    })
df_occ = pd.DataFrame(occ_records)

# --- Group summary ---
print("Bone occupancy by dataset (after fixed-FOV pipeline):")
print(f"{'='*64}")
grp = df_occ.groupby("dataset").agg(
    n=("volume_id", "count"),
    axial_occ_mean=("axial_occ", "mean"),
    axial_occ_std=("axial_occ", "std"),
    vol_occ_mean=("vol_occ", "mean"),
    spacing=("spacing_mm", "first"),
    fov=("phys_fov_mm", "first"),
).round(4)
display(grp)

healthy_axial = df_occ[df_occ["dataset"].isin(["healthy", "healthy_z"])]["axial_occ"].mean()
frac_axial = df_occ[df_occ["dataset"] == "fractured"]["axial_occ"].mean()
rel_diff = abs(frac_axial - healthy_axial) / max(healthy_axial, 1e-9)
print(f"\nHealthy mean axial occupancy : {healthy_axial:.1%}")
print(f"Fractured mean axial occupancy: {frac_axial:.1%}")
print(f"Relative difference          : {rel_diff:.1%}  "
      f"({'✅ within ±10%' if rel_diff <= 0.10 else '⚠️ exceeds ±10% — review'})")

n_spacings = df_occ['spacing_mm'].nunique()
print(f"Distinct output spacings     : {n_spacings}  "
      f"({'✅ constant' if n_spacings == 1 else '❌ not constant'})")

n_clipped = int(df_occ['clipped'].sum())
clipped_ids = df_occ[df_occ['clipped']]['volume_id'].tolist()
print(f"Volumes with trimmed anatomy : {n_clipped}  {clipped_ids if n_clipped else ''}")
print("  (trimming is expected/intended for long fractured shafts — verify the JOINT is kept below)")

# --- Occupancy distribution plot ---
fig, ax = plt.subplots(figsize=(10, 4))
for ds, color in [("healthy", "steelblue"), ("healthy_z", "seagreen"), ("fractured", "coral")]:
    vals = df_occ[df_occ["dataset"] == ds]["axial_occ"].values
    if len(vals):
        ax.scatter([ds] * len(vals), vals, color=color, alpha=0.6, s=40)
        ax.scatter([ds], [vals.mean()], color="black", marker="_", s=600, zorder=3)
ax.set_ylabel("Axial mid-slice bone occupancy")
ax.set_title("Axial Bone Occupancy by Dataset (black bar = mean)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# --- Knee-retention check: coronal view of every fractured case ---
frac_ids = df_occ[df_occ["dataset"] == "fractured"]["volume_id"].tolist()
if frac_ids:
    ncols = 4
    nrows = (len(frac_ids) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, vid in enumerate(frac_ids):
        arr, _ = _load_final(vid, "fractured")
        mid_y = arr.shape[1] // 2
        axes[i].imshow(arr[:, mid_y, :], cmap="gray", vmin=0, vmax=1, aspect="auto")
        clip = df_occ[df_occ["volume_id"] == vid]["clipped"].iloc[0]
        axes[i].set_title(f"{vid}{' [trimmed]' if clip else ''}", fontsize=9,
                          color="darkorange" if clip else "black")
        axes[i].axis("off")
    for j in range(len(frac_ids), len(axes)):
        axes[j].axis("off")
    plt.suptitle("Fractured cases — coronal view (confirm femoral condyles + tibial plateau / "
                 "joint line are RETAINED, not clipped)", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Metadata Summary
# ============================================================
df_meta = pd.read_csv(OUTPUT_DIR / "preprocessing_metadata.csv")

print(f"Preprocessing Metadata Summary")
print(f"{'='*60}")
print(f"Total volumes: {len(df_meta)}")
print()

# Per-dataset breakdown
for ds in ["healthy", "healthy_z", "fractured"]:
    subset = df_meta[df_meta["dataset"] == ds]
    if len(subset) == 0:
        continue
    n_clipped = int((subset[["clipped_z", "clipped_y", "clipped_x"]].any(axis=1)).sum())
    print(f"--- {ds} ({len(subset)} volumes) ---")
    print(f"  Spacing: {subset['output_spacing_mm'].min():.3f} – {subset['output_spacing_mm'].max():.3f} mm (constant)")
    print(f"  FOV:     {subset['phys_fov_mm'].min():.1f} – {subset['phys_fov_mm'].max():.1f} mm")
    print(f"  Trimmed (anatomy clipped to FOV): {n_clipped}/{len(subset)}")
    print(f"  HU range: [{subset['hu_min_raw'].min():.0f}, {subset['hu_max_raw'].max():.0f}]")
    print()

# Compact table
display(df_meta[["volume_id", "dataset", "crop_shape", "fov_voxels",
                  "clipped_z", "clipped_y", "clipped_x",
                  "output_spacing_mm", "phys_fov_mm"]])

## Summary & Next Steps

### Pipeline Applied
Load → Resample (0.5mm isotropic) → Orient (LPS) → Bone Window ([-450, 1050] HU) → **Body-Envelope Mask (drop CT table + extra body parts)** → ROI Bone Crop → **Center into Fixed Physical FOV (200mm cube)** → Resize to TARGET_SIZE³ → Save NIfTI

Fused-cast cases (`Case3`, `Case16`) are sourced from manually-cleaned NIfTIs; metal/TKR cases
(`Case8`; the LPS-verified TKR knees from `z050`/`z063`) and scouts (`Case4`, `Case10`) are excluded. The body-envelope mask
(step 4b) drops the CT table and extra body parts for every VSD and fractured volume.

### Why Fixed FOV (the bone-occupancy fix)
The previous `pad_to_cube` sized each output cube to the **longest** crop axis, so the per-volume
physical FOV and spacing varied. Fractured scans carry extra femur/tibia shaft (213–309mm Z) vs
the healthy ±100mm knee crop (~199mm Z), which inflated their cube → **smaller axial bone
occupancy** and **coarser effective resolution** (≈0.83–1.21mm vs ≈0.78mm). This biased DRR
generation: fractured DRRs would be lower-detail and at a different scale than healthy ones,
confounding the U-Net vs V-Net comparison and degrading sub-mm fracture detail.

**Fix:** every volume is centered into a fixed `FOV_MM = 200mm` cube before resize. Excess
fractured shaft is trimmed to the knee window; in-plane bone is preserved (≤~170mm < 200mm).

### Output
- **Location**: `data/interim/predrr_lps_256_v1/healthy/` and `data/interim/predrr_lps_256_v1/fractured/`
- **Format**: NIfTI, TARGET_SIZE³ voxels, intensity [0, 1]
- **Spacing**: **CONSTANT** `FOV_MM / TARGET_SIZE` mm isotropic for *all* volumes
  (200/256 = 0.78125mm on both local and HPC) — identical scale and resolution
  across healthy and fractured.
- **Metadata**: `preprocessing_metadata.csv` records `crop_shape`, `fov_mm`, `clipped_{z,y,x}`,
  constant `output_spacing_mm`.

### Verification (in this notebook)
- **Diagnostic cell**: fractured vs healthy axial bone occupancy comparable (within ~±10%), a
  single constant output spacing, and the knee joint retained (not clipped) after the 200mm trim.
- **All-Planes gallery**: every volume rendered superior-on-top (femur at the top) - confirms both
  the orientation correction and a consistent display convention.

### Next Steps
1. Generate bi-planar DRRs from these volumes using **DiffDRR** (`drr.ipynb`).
2. Because spacing/scale is now constant, DRR detector geometry can be shared across all cases.